In [2]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any




In [ ]:
class PetProfile:
    pet_id: str
    breed: str
    birth_date: datetime
    is_high_energy: bool  # e.g., Vizsla 
    postcode: str         # For seasonal/geographic tick risk 

@dataclass
class CheckInData:
    timestamp: datetime
    appetite: int          # 1-5 scale
    energy: int            # 1-5 scale
    sleep_quality: int     # 1-5 scale
    water_intake: str      # "Normal", "Increased" 
    toileting_blood: bool  # Acute override trigger 
    activity_minutes: int
    weight: int
    owner_struggling: bool # N/A for score, but flags dashboard 
    


class DoctorCheckIn:
    is_pain: bool
    dental_health: int
    skin_coat_parasite: list


In [ ]:
class VitalityScoreEngine:
    def __init__(self, pet: PetProfile, history: list):
        self.pet = pet
        self.history = history # Previous 4 weeks for baseline

    def calculate_score(self, current: CheckInData) -> Dict[str, Any]:
        # Check for Acute Overrides
        if current.toileting_blood or current.water_intake == "Increased":
            return {"score": 40, "zone": "Action Needed", "override": True}

        # Apply Breed-Based Weighting (set low, medium, high weight) 
        weights = {"activity": 0.2, "appetite": 0.25, "energy": 0.25, "sleep": 0.3}
        if self.pet.is_high_energy:
            weights["activity"] = 0.4 # Vizsla adjustment 
            weights["sleep"] = 0.1

        # Detect Signals (Combined symptoms) 
        raw_score = 100
        if current.energy < 3 and current.appetite < 3:
            raw_score -= 30  # "Moment 1" signal cluster 
        
        # Handle Baseline Period 
        if len(self.history) < 4:
            return {"score": None, "status": "Building your baseline"}

        return {"score": max(raw_score, 0), "zone": self._get_zone(raw_score)}

    def _get_zone(self, score):
        if score >= 85: return "Bright Green"
        if score >= 70: return "Medium Green"
        if score >= 50: return "Yellow/Watch Zone"
        return "Action Needed" 